# Cross-lingual NPI Probe — Analysis Notebook
### Does mBERT respect NPI licensing contexts in English and Spanish?

**Model:** `bert-base-multilingual-cased` (mBERT)  
**Phenomenon:** Negative Polarity Item (NPI) licensing  
**Languages:** English 🇬🇧 · Spanish 🇪🇸  

---

**What is an NPI?**  
NPIs are expressions (like *any*, *ever*, *yet* in English, or *nadie*, *nunca*, *jamás* in Spanish) that are grammatical **only** in certain *licensing* contexts, typically negation, polar questions, and conditionals:

| Licensed ✓ | Unlicensed ✗ |
|---|---|
| She doesn't have **any** money. | ~~She does have **any** money.~~ |
| No ha llamado **nadie**. | ~~Ha llamado **nadie**.~~ |

**Method:** Minimal pair scoring: we compare the log-probability mBERT assigns to the NPI token in the licensed vs. unlicensed context. Because the NPI word itself is identical in both sentences, Δ log p isolates the model's sensitivity to the licensing environment.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

BLUE   = '#4C72B0'
ORANGE = '#DD8452'
RED    = '#C44E52'
GREEN  = '#55A868'

## 1. Load results

In [ ]:
df  = pd.read_csv('../results/results_raw.csv')
acc = pd.read_csv('../results/results_accuracy.csv')

print(f"Total pairs scored : {len(df)}")
print(f"Overall accuracy   : {df['model_correct'].mean():.1%}")
print(f"Languages          : {df['language'].unique()}")
print(f"Licensor types     : {df['licensor'].unique()}")
df[['id','language','npi','licensor','target_npi','log_prob_diff','model_correct']].head(16)

## 2. Overall accuracy by language

In [ ]:
lang_acc = acc[acc['grouping'] == 'language'].copy()

fig, ax = plt.subplots(figsize=(5, 4))
colors = [BLUE if l == 'en' else ORANGE for l in lang_acc['language']]
bars = ax.bar(lang_acc['language'].str.upper(), lang_acc['accuracy'],
              color=colors, width=0.4, edgecolor='white')

for bar, val in zip(bars, lang_acc['accuracy']):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.03,
            f'{val:.0%}', ha='center', fontweight='bold')

ax.axhline(0.5, linestyle='--', color='gray', alpha=0.6, label='Chance (50%)')
ax.set_ylim(0, 1.15)
ax.set_ylabel('Accuracy')
ax.set_title('mBERT NPI Accuracy by Language', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../results/accuracy_by_language.png', bbox_inches='tight')
plt.show()

## 3. Accuracy by licensor type

This is the most linguistically interesting breakdown. NPI theory predicts that negation is the strongest licensor, does mBERT reflect this? Are questions and conditionals harder?

In [ ]:
lic_acc = acc[acc['grouping'] == 'licensor'].copy()
lic_acc = lic_acc.sort_values('accuracy', ascending=True)

fig, ax = plt.subplots(figsize=(7, 4))
colors = [GREEN if v >= 0.5 else RED for v in lic_acc['accuracy']]
bars = ax.barh(lic_acc['licensor'].str.replace('_', ' '), lic_acc['accuracy'],
               color=colors, edgecolor='white')

for bar, val in zip(bars, lic_acc['accuracy']):
    ax.text(val + 0.02, bar.get_y() + bar.get_height()/2,
            f'{val:.0%}', va='center', fontweight='bold')

ax.axvline(0.5, linestyle='--', color='gray', alpha=0.6, label='Chance (50%)')
ax.set_xlim(0, 1.15)
ax.set_xlabel('Accuracy')
ax.set_title('mBERT Accuracy by NPI Licensor Type', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../results/accuracy_by_licensor.png', bbox_inches='tight')
plt.show()

## 4. Cross-lingual heatmap: Language × Licensor

Where does EN and ES diverge? Does mBERT handle long-distance licensing differently across languages?

In [ ]:
cross = acc[acc['grouping'] == 'lang×lic'].copy()
pivot = cross.pivot(index='licensor', columns='language', values='accuracy')

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(pivot, annot=True, fmt='.0%', cmap='RdYlGn',
            vmin=0, vmax=1, linewidths=0.5, ax=ax,
            annot_kws={'fontsize': 12, 'fontweight': 'bold'})

ax.set_title('Accuracy: Language × Licensor', fontweight='bold')
ax.set_xlabel('Language')
ax.set_ylabel('Licensor type')
ax.set_yticklabels([t.get_text().replace('_', ' ') for t in ax.get_yticklabels()], rotation=0)
plt.tight_layout()
plt.savefig('../results/heatmap_lang_x_licensor.png', bbox_inches='tight')
plt.show()

## 5. Log-probability differences per pair

Δ log p = log p(NPI in licensed context) − log p(NPI in unlicensed context)  
- **Positive (blue):** model correctly prefers the licensed context  
- **Negative (red):** model incorrectly prefers the unlicensed context

In [ ]:
df_sorted = df.sort_values(['language', 'licensor'])

fig, ax = plt.subplots(figsize=(10, 6))
colors = [BLUE if c else RED for c in df_sorted['model_correct']]
labels = [f"{r['id']}\n({r['npi']} | {r['licensor'].replace('_',' ')})" 
          for _, r in df_sorted.iterrows()]

ax.barh(labels, df_sorted['log_prob_diff'], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Δ log p  (licensed − unlicensed)')
ax.set_title('Model Preference per NPI Pair', fontweight='bold')

correct_patch = mpatches.Patch(color=BLUE, label='Model correct (Δ > 0)')
wrong_patch   = mpatches.Patch(color=RED,  label='Model incorrect (Δ < 0)')
ax.legend(handles=[correct_patch, wrong_patch])

plt.tight_layout()
plt.savefig('../results/logprob_diff_per_pair.png', bbox_inches='tight')
plt.show()

## 6. Linguistic Analysis

Use this section to write your interpretation. Some guiding questions:

### 6.1 Does mBERT understand NPI licensing at all?
Overall accuracy above 50% is above chance, but the interesting question is *how far* above — and whether it varies by condition.

### 6.2 Which licensor type is hardest?
NPI theory (Ladusaw 1980, Giannakidou 1998) predicts that **negation** is the canonical and strongest licensor. Do the results reflect this? Is long-distance licensing harder than local negation?

### 6.3 Cross-lingual comparison: EN vs ES
Spanish has **negative concord** — you can say *No vi a nadie* ("I didn't see nobody") where multiple negative elements co-occur. English disallows this. Does mBERT pick up on this structural difference? Are Spanish NPIs easier or harder to probe?

### 6.4 Strong vs weak NPIs
- **Weak NPIs** (*any*, *ever*) are licensed in all downward-entailing environments
- **Strong NPIs** (*at all*, *en absoluto*, *jamás*) require strict negation

Does mBERT show a difference in confidence (Δ log p magnitude) between strong and weak NPIs?

### 6.5 Long-distance licensing
The most interesting pairs are `en_npi_08` and `es_npi_07`, where negation in a matrix clause licenses an NPI in an embedded clause. This requires the model to track dependencies across a clause boundary. Does it?

---

*Write your analysis here...*